# Week 13: RNN / Sequence Modeling (LSTM/GRU; Transformers Concepts)
# 第 13 週：RNN / 序列建模（LSTM/GRU；Transformers 概念）

## 學習目標 Learning Objectives
1. 理解序列資料的特性，使用 numpy 生成與視覺化時間序列
2. 從零實作 Vanilla RNN Cell，觀察隱藏狀態演化
3. 視覺化梯度消失問題（連乘矩陣的梯度衰減）
4. 繪製 LSTM 三門控結構圖，模擬門控激活值
5. 比較 GRU 與 LSTM 的結構差異
6. 使用 sklearn 在手工序列特徵上進行時間序列分類
7. 從零實作點積注意力機制，視覺化注意力權重
8. 實作 Scaled Dot-Product Self-Attention (Q, K, V)
9. 繪製 Transformer 架構概覽圖

**環境需求 Required packages**：numpy, matplotlib, sklearn, scipy

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.special import expit as sigmoid  # stable sigmoid

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# 設定隨機種子
np.random.seed(42)

print('所有套件匯入成功！ All packages imported successfully!')

---
## Part 1: 序列資料的類型與生成 Sequence Data Types

序列資料是指資料點之間具有**時間順序**或**位置順序**依賴關係的資料。
常見類型包括：時間序列 (Time Series)、自然語言 (Natural Language)、語音訊號等。

我們先生成並視覺化幾種典型的序列資料。

In [ ]:
# 生成不同類型的時間序列 Generate different time series types
np.random.seed(42)
t = np.linspace(0, 4 * np.pi, 200)

# 1. 正弦波 + 噪音
sine_wave = np.sin(t) + 0.3 * np.random.randn(len(t))

# 2. 帶趨勢的序列
trend_series = 0.05 * t + np.sin(t) + 0.2 * np.random.randn(len(t))

# 3. 方波序列
square_wave = np.sign(np.sin(t)) + 0.2 * np.random.randn(len(t))

# 4. 隨機遊走 (Random Walk)
random_walk = np.cumsum(np.random.randn(len(t)) * 0.1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

series_data = [
    (sine_wave, 'Sine Wave + Noise (正弦波+噪音)', 'royalblue'),
    (trend_series, 'Trend + Periodic (趨勢+週期)', 'darkorange'),
    (square_wave, 'Square Wave + Noise (方波+噪音)', 'forestgreen'),
    (random_walk, 'Random Walk (隨機遊走)', 'crimson')
]

for ax, (data, title, color) in zip(axes.flat, series_data):
    ax.plot(t, data, color=color, linewidth=1.2, alpha=0.9)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Time', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Common Sequence Data Types 常見序列資料類型', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 文字序列範例 Text sequence example
print('\n=== 文字序列範例 Text Sequence Example ===')
sentence = 'The cat sat on the mat because it was tired'
tokens = sentence.split()
print(f'Sentence: {sentence}')
print(f'Tokens: {tokens}')
print(f'Sequence length: {len(tokens)}')
print(f'Vocabulary: {sorted(set(tokens))}')

---
## Part 2: 從零實作 Vanilla RNN Simple RNN from Scratch

RNN 的核心思想是引入**隱藏狀態 (Hidden State)**，讓網路在處理序列時「記住」之前的資訊。

**隱藏狀態更新公式：**
$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

**輸出計算：**
$$y_t = W_{hy} \cdot h_t + b_y$$

參數 $W_{xh}, W_{hh}, W_{hy}$ 在所有時間步上**共享 (Parameter Sharing)**。

In [ ]:
class VanillaRNNCell:
    """
    從零實作的 Vanilla RNN Cell
    Vanilla RNN Cell implemented from scratch with numpy.
    """
    def __init__(self, input_dim, hidden_dim, output_dim):
        np.random.seed(42)
        scale = 0.1
        self.hidden_dim = hidden_dim
        # 權重初始化 Weight initialization
        self.W_xh = np.random.randn(hidden_dim, input_dim) * scale
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.W_hy = np.random.randn(output_dim, hidden_dim) * scale
        self.b_h = np.zeros((hidden_dim, 1))
        self.b_y = np.zeros((output_dim, 1))

    def forward(self, x_sequence):
        """
        前向傳播：逐步處理序列 Process sequence step by step.
        x_sequence: (seq_len, input_dim)
        """
        seq_len = x_sequence.shape[0]
        h = np.zeros((self.hidden_dim, 1))  # 初始隱藏狀態
        
        hidden_states = [h.copy()]
        outputs = []
        
        for t in range(seq_len):
            x_t = x_sequence[t].reshape(-1, 1)  # (input_dim, 1)
            # 隱藏狀態更新 Hidden state update
            h = np.tanh(self.W_xh @ x_t + self.W_hh @ h + self.b_h)
            # 輸出計算 Output computation
            y_t = self.W_hy @ h + self.b_y
            
            hidden_states.append(h.copy())
            outputs.append(y_t.copy())
        
        return np.array(hidden_states), np.array(outputs)

# 建立 RNN 並處理正弦波序列
input_dim = 1
hidden_dim = 8
output_dim = 1

rnn = VanillaRNNCell(input_dim, hidden_dim, output_dim)

# 使用正弦波作為輸入
t_short = np.linspace(0, 2 * np.pi, 50)
x_input = np.sin(t_short).reshape(-1, 1)  # (50, 1)

hidden_states, outputs = rnn.forward(x_input)

print(f'Input sequence length: {len(t_short)}')
print(f'Hidden state dimension: {hidden_dim}')
print(f'Number of hidden states recorded: {len(hidden_states)}')
print(f'Hidden state shape at each step: {hidden_states[0].shape}')

In [ ]:
# 視覺化隱藏狀態演化 Visualize hidden state evolution
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 上圖：輸入序列
axes[0].plot(t_short, x_input, 'b-', linewidth=2, label='Input: sin(t)')
axes[0].set_ylabel('Input Value', fontsize=11)
axes[0].set_title('RNN Forward Pass: Input, Hidden States, Output', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 中圖：隱藏狀態熱力圖
h_matrix = np.array([h.flatten() for h in hidden_states[1:]])  # (seq_len, hidden_dim)
im = axes[1].imshow(h_matrix.T, aspect='auto', cmap='RdBu_r',
                     interpolation='nearest', vmin=-1, vmax=1)
axes[1].set_ylabel('Hidden Dimension', fontsize=11)
axes[1].set_xlabel('Time Step', fontsize=11)
axes[1].set_title('Hidden State Evolution (隱藏狀態演化)', fontsize=12)
plt.colorbar(im, ax=axes[1], label='Activation Value')

# 下圖：選取幾個隱藏維度的演化曲線
for dim_idx in [0, 2, 4, 6]:
    axes[2].plot(t_short, h_matrix[:, dim_idx], linewidth=1.5,
                 label=f'h[{dim_idx}]', alpha=0.8)
axes[2].set_xlabel('Time', fontsize=11)
axes[2].set_ylabel('Hidden Value', fontsize=11)
axes[2].set_title('Selected Hidden Dimensions Over Time (部分隱藏維度隨時間變化)', fontsize=12)
axes[2].legend(fontsize=9, ncol=4)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('觀察重點 Key Observations:')
print('1. 每個隱藏維度以不同方式回應輸入序列')
print('2. 隱藏狀態保留了過去輸入的「記憶」')
print('3. tanh 將所有值限制在 [-1, 1] 區間')

---
## Part 3: 梯度消失問題 Vanishing Gradient Problem

RNN 的梯度需要沿時間步逐步傳播。根據鏈式法則：

$$\frac{\partial L_T}{\partial h_1} = \frac{\partial L_T}{\partial h_T} \cdot \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

由於每一步都要乘以 $W_{hh}$ 和 $\tanh'(\cdot)$ 的對角矩陣：
- 若 $W_{hh}$ 的最大特徵值 < 1 → 梯度**指數衰減** (Vanishing)
- 若 $W_{hh}$ 的最大特徵值 > 1 → 梯度**指數增長** (Exploding)

In [ ]:
# 展示梯度消失/爆炸問題
# Demonstrate vanishing/exploding gradient through repeated matrix multiplication
np.random.seed(42)
hidden_size = 10
num_steps = 50

# 不同特徵值尺度的權重矩陣
scenarios = {
    'Vanishing (max eigenvalue < 1)': 0.5,
    'Stable (max eigenvalue ~ 1)': 1.0,
    'Exploding (max eigenvalue > 1)': 1.5
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (scenario_name, scale) in enumerate(scenarios.items()):
    # 建立權重矩陣並縮放其特徵值
    W = np.random.randn(hidden_size, hidden_size)
    # SVD 分解後縮放奇異值，控制最大特徵值
    U, s, Vt = np.linalg.svd(W)
    s = s / s.max() * scale  # 控制最大奇異值
    W_scaled = U @ np.diag(s) @ Vt
    
    # 模擬梯度沿時間傳播：連續乘以 diag(tanh') * W_hh
    gradient_norms = []
    grad = np.eye(hidden_size)  # 初始梯度
    
    for step in range(num_steps):
        # 模擬 tanh 導數（值在 [0, 1] 之間）
        tanh_deriv = np.random.uniform(0.2, 0.8, hidden_size)
        jacobian = np.diag(tanh_deriv) @ W_scaled
        grad = jacobian @ grad
        gradient_norms.append(np.linalg.norm(grad))
    
    ax = axes[idx]
    ax.semilogy(range(1, num_steps + 1), gradient_norms, 'o-',
                markersize=3, linewidth=1.5)
    ax.set_xlabel('Time Steps Back (往回傳播步數)', fontsize=10)
    ax.set_ylabel('Gradient Norm (log scale)', fontsize=10)
    ax.set_title(scenario_name, fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # 標示最大特徵值
    max_eig = np.max(np.abs(np.linalg.eigvals(W_scaled)))
    ax.text(0.95, 0.95, f'|max eigenvalue| = {max_eig:.2f}',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            fontsize=9)

plt.suptitle('Gradient Flow Through RNN: Vanishing vs Exploding\n'
             'RNN 梯度流：消失 vs 爆炸', fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

print('梯度消失的後果 Consequences of vanishing gradients:')
print('  - 模型無法學到長程依賴 (Long-Range Dependencies)')
print('  - 早期時間步的權重幾乎不會被更新')
print('  - 例："The cat, which ... was full" 中 was 無法關聯到 cat')
print('\n解決方案：LSTM 和 GRU 透過門控機制緩解此問題。')

---
## Part 4: LSTM 門控結構視覺化 LSTM Cell Visualization

LSTM 引入了**細胞狀態 (Cell State)** 和三個門控：
- **遺忘門 Forget Gate** $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
- **輸入門 Input Gate** $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
- **輸出門 Output Gate** $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$

細胞狀態更新（加法！不是乘法）：
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

這個**加法**更新是 LSTM 能緩解梯度消失的關鍵：梯度可以幾乎不衰減地流過。

In [ ]:
# 繪製 LSTM 門控結構圖 Draw LSTM gate structure diagram
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')

# 主體框
main_box = FancyBboxPatch((2, 1.5), 12, 7, boxstyle='round,pad=0.3',
                           facecolor='lightyellow', edgecolor='black', linewidth=2)
ax.add_patch(main_box)
ax.text(8, 9.2, 'LSTM Cell', fontsize=16, fontweight='bold', ha='center')

# 門控圓圈
gate_props = dict(fontsize=11, fontweight='bold', ha='center', va='center')

# Forget Gate
forget_circle = plt.Circle((4, 3.5), 0.6, color='#FF6B6B', alpha=0.8)
ax.add_patch(forget_circle)
ax.text(4, 3.5, '$\sigma$', **gate_props, color='white')
ax.text(4, 2.5, 'Forget\nGate $f_t$', fontsize=9, ha='center', color='#CC3333')

# Input Gate
input_circle = plt.Circle((6.5, 3.5), 0.6, color='#4ECDC4', alpha=0.8)
ax.add_patch(input_circle)
ax.text(6.5, 3.5, '$\sigma$', **gate_props, color='white')
ax.text(6.5, 2.5, 'Input\nGate $i_t$', fontsize=9, ha='center', color='#2B8C85')

# Candidate memory
cand_circle = plt.Circle((9, 3.5), 0.6, color='#45B7D1', alpha=0.8)
ax.add_patch(cand_circle)
ax.text(9, 3.5, 'tanh', fontsize=9, fontweight='bold', ha='center', va='center', color='white')
ax.text(9, 2.5, 'Candidate\n$\\tilde{C}_t$', fontsize=9, ha='center', color='#2980B9')

# Output Gate
output_circle = plt.Circle((12, 3.5), 0.6, color='#F39C12', alpha=0.8)
ax.add_patch(output_circle)
ax.text(12, 3.5, '$\sigma$', **gate_props, color='white')
ax.text(12, 2.5, 'Output\nGate $o_t$', fontsize=9, ha='center', color='#D68910')

# Cell state line (highway)
ax.annotate('', xy=(14, 7), xytext=(2, 7),
            arrowprops=dict(arrowstyle='->', lw=3, color='green'))
ax.text(1.5, 7.3, '$C_{t-1}$', fontsize=12, fontweight='bold', color='green')
ax.text(14.2, 7.3, '$C_t$', fontsize=12, fontweight='bold', color='green')
ax.text(8, 7.5, 'Cell State (細胞狀態高速公路)', fontsize=10,
        ha='center', color='green', fontstyle='italic')

# Multiply (forget) symbol
ax.plot(4, 7, 'x', markersize=15, color='#CC3333', markeredgewidth=3)
ax.annotate('', xy=(4, 6.6), xytext=(4, 4.1),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#CC3333'))

# Add (input) symbol
ax.plot(7.5, 7, '+', markersize=20, color='#2B8C85', markeredgewidth=3)
# Connection from input gate * candidate
ax.annotate('', xy=(7.5, 6.6), xytext=(7.5, 5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#2B8C85'))
# Multiply symbol for i_t * C_tilde
ax.plot(7.5, 5, 'x', markersize=12, color='#2B8C85', markeredgewidth=2)
ax.annotate('', xy=(7.1, 4.7), xytext=(6.9, 4.1),
            arrowprops=dict(arrowstyle='->', lw=1, color='#2B8C85'))
ax.annotate('', xy=(7.9, 4.7), xytext=(8.6, 4.1),
            arrowprops=dict(arrowstyle='->', lw=1, color='#45B7D1'))

# Output path: tanh(C_t) * o_t -> h_t
tanh_out = plt.Circle((12, 6), 0.4, color='#45B7D1', alpha=0.6)
ax.add_patch(tanh_out)
ax.text(12, 6, 'tanh', fontsize=7, ha='center', va='center', fontweight='bold')
ax.annotate('', xy=(12, 6.6), xytext=(12, 7),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='green'))
ax.plot(12, 5, 'x', markersize=12, color='#D68910', markeredgewidth=2)
ax.annotate('', xy=(12, 5.6), xytext=(12, 5.4),
            arrowprops=dict(arrowstyle='->', lw=1, color='#45B7D1'))
ax.annotate('', xy=(12, 4.7), xytext=(12, 4.1),
            arrowprops=dict(arrowstyle='->', lw=1, color='#D68910'))
ax.text(13.5, 5, '$h_t$', fontsize=14, fontweight='bold')
ax.annotate('', xy=(13, 5), xytext=(12.3, 5),
            arrowprops=dict(arrowstyle='->', lw=2))

# Input arrows
ax.text(8, 1.2, '$x_t$ (Input) + $h_{t-1}$ (Previous Hidden State)', fontsize=10,
        ha='center', fontstyle='italic')

plt.title('LSTM Cell Architecture\nLSTM 門控結構圖', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print('LSTM 核心洞察 Key Insight:')
print('  Cell State 的更新是加法 (Additive)，梯度可以幾乎不衰減地流過！')
print('  這與 ResNet 的殘差連接 (Residual Connection) 原理相似。')

In [ ]:
# 模擬 LSTM 門控激活值 Simulate LSTM gate activations
np.random.seed(42)
seq_len = 60
hidden_dim_lstm = 4
input_dim_lstm = 1

# 建立一個包含不同模式的輸入序列
t_lstm = np.linspace(0, 4 * np.pi, seq_len)
x_lstm = np.sin(t_lstm)  # 正弦輸入

# 簡化的 LSTM 前向傳播
W_f = np.random.randn(hidden_dim_lstm, input_dim_lstm + hidden_dim_lstm) * 0.3
W_i = np.random.randn(hidden_dim_lstm, input_dim_lstm + hidden_dim_lstm) * 0.3
W_c = np.random.randn(hidden_dim_lstm, input_dim_lstm + hidden_dim_lstm) * 0.3
W_o = np.random.randn(hidden_dim_lstm, input_dim_lstm + hidden_dim_lstm) * 0.3
b_f = np.ones((hidden_dim_lstm, 1)) * 1.0  # 偏向「記住」
b_i = np.zeros((hidden_dim_lstm, 1))
b_c = np.zeros((hidden_dim_lstm, 1))
b_o = np.zeros((hidden_dim_lstm, 1))

h = np.zeros((hidden_dim_lstm, 1))
C = np.zeros((hidden_dim_lstm, 1))

forget_gates = []
input_gates = []
output_gates = []
cell_states = []
hidden_out = []

for t_step in range(seq_len):
    x_t = np.array([[x_lstm[t_step]]])
    concat = np.vstack([h, x_t])  # [h_{t-1}; x_t]
    
    f_t = sigmoid(W_f @ concat + b_f)
    i_t = sigmoid(W_i @ concat + b_i)
    C_tilde = np.tanh(W_c @ concat + b_c)
    o_t = sigmoid(W_o @ concat + b_o)
    
    C = f_t * C + i_t * C_tilde
    h = o_t * np.tanh(C)
    
    forget_gates.append(f_t.flatten())
    input_gates.append(i_t.flatten())
    output_gates.append(o_t.flatten())
    cell_states.append(C.flatten())
    hidden_out.append(h.flatten())

forget_gates = np.array(forget_gates)
input_gates = np.array(input_gates)
output_gates = np.array(output_gates)
cell_states = np.array(cell_states)

# 視覺化門控激活值
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)

axes[0].plot(t_lstm, x_lstm, 'k-', linewidth=2)
axes[0].set_ylabel('Input', fontsize=10)
axes[0].set_title('LSTM Gate Activations Over Time\nLSTM 門控激活值隨時間變化', fontsize=13)
axes[0].grid(True, alpha=0.3)

im1 = axes[1].imshow(forget_gates.T, aspect='auto', cmap='Reds', vmin=0, vmax=1)
axes[1].set_ylabel('Forget Gate\n$f_t$', fontsize=10)
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(input_gates.T, aspect='auto', cmap='Greens', vmin=0, vmax=1)
axes[2].set_ylabel('Input Gate\n$i_t$', fontsize=10)
plt.colorbar(im2, ax=axes[2])

im3 = axes[3].imshow(output_gates.T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
axes[3].set_ylabel('Output Gate\n$o_t$', fontsize=10)
plt.colorbar(im3, ax=axes[3])

im4 = axes[4].imshow(cell_states.T, aspect='auto', cmap='RdBu_r')
axes[4].set_ylabel('Cell State\n$C_t$', fontsize=10)
axes[4].set_xlabel('Time Step', fontsize=11)
plt.colorbar(im4, ax=axes[4])

plt.tight_layout()
plt.show()

print('觀察重點 Key Observations:')
print('1. Forget Gate 值接近 1 時 → 保留舊記憶')
print('2. Input Gate 控制寫入新資訊的強度')
print('3. Cell State 可以累積/保持長期資訊')
print('4. Output Gate 決定哪些記憶被「展示」')

---
## Part 5: GRU 結構與 LSTM 比較 GRU Cell Visualization

GRU (Gated Recurrent Unit) 是 LSTM 的**簡化版本**，只有兩個門：
- **重置門 Reset Gate** $r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)$
- **更新門 Update Gate** $z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)$

$$\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t] + b_h)$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

GRU 合併了細胞狀態與隱藏狀態，參數更少。

In [ ]:
# LSTM vs GRU 結構比較圖
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# === LSTM ===
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('LSTM Cell', fontsize=14, fontweight='bold')

# Main box
ax.add_patch(FancyBboxPatch((0.5, 0.5), 9, 8.5, boxstyle='round,pad=0.2',
                             facecolor='#FFF3E0', edgecolor='black', linewidth=2))

# Gates
lstm_gates = [
    (2, 3, '#FF6B6B', 'Forget\n$f_t$'),
    (4, 3, '#4ECDC4', 'Input\n$i_t$'),
    (6, 3, '#45B7D1', 'Candidate\n$\\tilde{C}_t$'),
    (8, 3, '#F39C12', 'Output\n$o_t$')
]
for x, y, color, label in lstm_gates:
    ax.add_patch(plt.Circle((x, y), 0.55, color=color, alpha=0.8))
    ax.text(x, y - 1, label, fontsize=8, ha='center', fontweight='bold')

# State lines
ax.annotate('', xy=(9, 7), xytext=(1, 7),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='green'))
ax.text(5, 7.5, 'Cell State $C_t$ (長期記憶)', fontsize=9, ha='center', color='green')
ax.annotate('', xy=(9, 5), xytext=(1, 5),
            arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
ax.text(5, 5.5, 'Hidden State $h_t$ (短期記憶)', fontsize=9, ha='center', color='blue')

# Stats
stats_text = 'Gates: 3 (f, i, o)\nStates: 2 ($C_t$, $h_t$)\nParams: ~4$n_h^2$'
ax.text(5, 1.2, stats_text, fontsize=9, ha='center',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# === GRU ===
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('GRU Cell', fontsize=14, fontweight='bold')

# Main box
ax.add_patch(FancyBboxPatch((0.5, 0.5), 9, 8.5, boxstyle='round,pad=0.2',
                             facecolor='#E8F5E9', edgecolor='black', linewidth=2))

# Gates
gru_gates = [
    (3, 3, '#9B59B6', 'Reset\n$r_t$'),
    (5, 3, '#E74C3C', 'Update\n$z_t$'),
    (7, 3, '#3498DB', 'Candidate\n$\\tilde{h}_t$')
]
for x, y, color, label in gru_gates:
    ax.add_patch(plt.Circle((x, y), 0.55, color=color, alpha=0.8))
    ax.text(x, y - 1, label, fontsize=8, ha='center', fontweight='bold')

# Single state line
ax.annotate('', xy=(9, 6), xytext=(1, 6),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='purple'))
ax.text(5, 6.5, 'Hidden State $h_t$ (合併記憶)', fontsize=9, ha='center', color='purple')

# Key equation
ax.text(5, 8, '$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \\tilde{h}_t$',
        fontsize=11, ha='center', fontstyle='italic',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# Stats
stats_text = 'Gates: 2 (r, z)\nStates: 1 ($h_t$)\nParams: ~3$n_h^2$'
ax.text(5, 1.2, stats_text, fontsize=9, ha='center',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('LSTM vs GRU: Structural Comparison\nLSTM vs GRU 結構比較',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 表格比較
print('=' * 60)
print(f'{"Feature":<25} {"LSTM":<18} {"GRU":<18}')
print('=' * 60)
comparisons = [
    ('Number of Gates', '3 (f, i, o)', '2 (r, z)'),
    ('State Variables', '2 (h_t, C_t)', '1 (h_t)'),
    ('Parameter Count', '~4 * n_h^2', '~3 * n_h^2'),
    ('Long-range Memory', 'Excellent', 'Good'),
    ('Training Speed', 'Slower', 'Faster'),
    ('Year Proposed', '1997', '2014'),
]
for feature, lstm, gru in comparisons:
    print(f'{feature:<25} {lstm:<18} {gru:<18}')
print('=' * 60)

---
## Part 6: 序列分類 Demo Sequence Classification Demo

雖然我們不使用深度學習框架，但可以透過**手工提取序列特徵 (Feature Engineering)**，
然後使用 sklearn 分類器來完成時間序列分類任務。

特徵包括：滾動平均 (Rolling Mean)、趨勢斜率、波動性 (Volatility)、自相關等。

In [ ]:
# 生成三類時間序列模式 Generate three classes of time series patterns
np.random.seed(42)
n_samples_per_class = 100
seq_length = 100

def generate_time_series(pattern, n_samples, length):
    """生成指定模式的時間序列 Generate time series of a given pattern."""
    series_list = []
    t = np.linspace(0, 4 * np.pi, length)
    for _ in range(n_samples):
        noise = np.random.randn(length) * 0.3
        if pattern == 'sine':
            freq = np.random.uniform(0.8, 1.2)
            s = np.sin(freq * t) + noise
        elif pattern == 'trend_up':
            slope = np.random.uniform(0.02, 0.05)
            s = slope * np.arange(length) + noise
        elif pattern == 'random_walk':
            s = np.cumsum(np.random.randn(length) * 0.2)
        series_list.append(s)
    return np.array(series_list)

# 生成三類
sine_data = generate_time_series('sine', n_samples_per_class, seq_length)
trend_data = generate_time_series('trend_up', n_samples_per_class, seq_length)
walk_data = generate_time_series('random_walk', n_samples_per_class, seq_length)

# 組合並建立標籤
X_ts = np.vstack([sine_data, trend_data, walk_data])
y_ts = np.array([0] * n_samples_per_class + [1] * n_samples_per_class + [2] * n_samples_per_class)
class_names = ['Sine Wave', 'Trend Up', 'Random Walk']

# 視覺化範例
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (name, data) in enumerate(zip(class_names, [sine_data, trend_data, walk_data])):
    for j in range(5):
        axes[i].plot(data[j], alpha=0.5, linewidth=1)
    axes[i].set_title(f'Class {i}: {name}', fontsize=12)
    axes[i].set_xlabel('Time Step')
    axes[i].grid(True, alpha=0.3)
plt.suptitle('Time Series Classification: Three Patterns\n時間序列分類：三種模式', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Total samples: {len(X_ts)}')
print(f'Sequence length: {seq_length}')
print(f'Classes: {class_names}')

In [ ]:
# 手工提取序列特徵 Extract hand-crafted features
def extract_features(series):
    """
    從時間序列中提取統計特徵
    Extract statistical features from a time series.
    """
    features = []
    features.append(np.mean(series))          # 平均值
    features.append(np.std(series))            # 標準差
    features.append(np.max(series) - np.min(series))  # 範圍
    features.append(series[-1] - series[0])    # 整體趨勢
    
    # 滾動平均差異 Rolling mean difference
    first_half_mean = np.mean(series[:len(series)//2])
    second_half_mean = np.mean(series[len(series)//2:])
    features.append(second_half_mean - first_half_mean)
    
    # 波動性（差分標準差）Volatility
    features.append(np.std(np.diff(series)))
    
    # 自相關 (lag=1) Autocorrelation
    normalized = (series - np.mean(series)) / (np.std(series) + 1e-8)
    autocorr = np.correlate(normalized, normalized, mode='full')
    autocorr = autocorr[len(autocorr)//2:]
    autocorr = autocorr / autocorr[0]
    features.append(autocorr[1] if len(autocorr) > 1 else 0)
    
    # 零交叉次數 Zero-crossing rate
    zero_crossings = np.sum(np.abs(np.diff(np.sign(series - np.mean(series)))) > 0)
    features.append(zero_crossings / len(series))
    
    # 線性擬合斜率 Linear fit slope
    slope = np.polyfit(np.arange(len(series)), series, 1)[0]
    features.append(slope)
    
    return np.array(features)

# 提取所有樣本的特徵
feature_names = ['Mean', 'Std', 'Range', 'Trend', 'Half_Diff',
                 'Volatility', 'Autocorr', 'Zero_Cross', 'Slope']
X_features = np.array([extract_features(s) for s in X_ts])

print(f'Feature matrix shape: {X_features.shape}')
print(f'Feature names: {feature_names}')

# 訓練分類器
X_train_ts, X_test_ts, y_train_ts, y_test_ts = train_test_split(
    X_features, y_ts, test_size=0.3, random_state=42, stratify=y_ts
)

scaler_ts = StandardScaler()
X_train_ts = scaler_ts.fit_transform(X_train_ts)
X_test_ts = scaler_ts.transform(X_test_ts)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_ts, y_train_ts)

y_pred_ts = clf.predict(X_test_ts)
acc = accuracy_score(y_test_ts, y_pred_ts)

print(f'\nClassification Accuracy: {acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test_ts, y_pred_ts, target_names=class_names))

# 特徵重要度
importances = clf.feature_importances_
fig, ax = plt.subplots(figsize=(10, 4))
sorted_idx = np.argsort(importances)[::-1]
ax.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx],
        color='steelblue', alpha=0.8)
ax.set_xlabel('Importance', fontsize=11)
ax.set_title('Feature Importance for Time Series Classification\n特徵重要度', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## Part 7: 注意力機制 Attention Mechanism Concept

注意力機制的核心想法是讓 Decoder 在每個時間步**回頭看** Encoder 的所有隱藏狀態，
並根據當前需求**選擇性地關注**最相關的部分。

**計算步驟：**
1. 計算對齊分數：$e_{t,i} = \text{score}(s_{t-1}, h_i)$
2. Softmax 正規化：$\alpha_{t,i} = \text{softmax}(e_t)_i$
3. 加權求和：$c_t = \sum_i \alpha_{t,i} \cdot h_i$

In [ ]:
# 從零實作點積注意力 Implement dot-product attention from scratch
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def dot_product_attention(query, keys, values):
    """
    點積注意力機制 Dot-product attention.
    query:  (d,)   - decoder hidden state at time t
    keys:   (T, d) - encoder hidden states
    values: (T, d) - encoder hidden states (same as keys in basic attention)
    Returns: context vector, attention weights
    """
    # 計算對齊分數 Alignment scores
    scores = keys @ query  # (T,)
    # Softmax 正規化
    weights = softmax(scores)  # (T,)
    # 加權求和 Weighted sum
    context = weights @ values  # (d,)
    return context, weights

# 模擬翻譯場景：Encoder 有 6 個時間步
np.random.seed(42)
encoder_len = 6
d_model = 8
decoder_len = 4

# 模擬 encoder 隱藏狀態 (source sentence hidden states)
encoder_states = np.random.randn(encoder_len, d_model)
# 模擬 decoder 隱藏狀態 (target sentence hidden states)
decoder_states = np.random.randn(decoder_len, d_model)

source_words = ['I', 'am', 'a', 'good', 'student', '.']
target_words = ['我', '是', '好', '學生']

# 計算每個 decoder 步的注意力
all_weights = []
for t in range(decoder_len):
    _, weights = dot_product_attention(
        decoder_states[t], encoder_states, encoder_states
    )
    all_weights.append(weights)

attention_matrix = np.array(all_weights)

# 視覺化注意力權重熱力圖
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(attention_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(encoder_len))
ax.set_xticklabels(source_words, fontsize=12)
ax.set_yticks(range(decoder_len))
ax.set_yticklabels(target_words, fontsize=12)
ax.set_xlabel('Source (Encoder) Words', fontsize=12)
ax.set_ylabel('Target (Decoder) Words', fontsize=12)
ax.set_title('Attention Weights Heatmap (注意力權重熱力圖)\n'
             'Simulated English-to-Chinese Translation', fontsize=13)

# 在每格顯示數值
for i in range(decoder_len):
    for j in range(encoder_len):
        ax.text(j, i, f'{attention_matrix[i, j]:.2f}',
                ha='center', va='center', fontsize=10,
                color='white' if attention_matrix[i, j] > 0.3 else 'black')

plt.colorbar(im, label='Attention Weight')
plt.tight_layout()
plt.show()

print('注意力機制的核心價值 Core Value of Attention:')
print('  - 打破了固定長度上下文向量的瓶頸')
print('  - 讓模型學會「看哪裡」而非壓縮所有資訊')
print('  - 注意力權重提供了可解釋性')

---
## Part 8: Self-Attention 自注意力機制 (Q, K, V)

Self-Attention 讓序列中的**每個位置**都能直接關注**所有其他位置**。

**Scaled Dot-Product Attention：**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

其中 $Q = XW^Q$, $K = XW^K$, $V = XW^V$。

In [ ]:
# 實作 Scaled Dot-Product Self-Attention
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention.
    Q: (seq_len, d_k)
    K: (seq_len, d_k)
    V: (seq_len, d_v)
    Returns: output (seq_len, d_v), attention_weights (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    # 計算注意力分數 Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)
    
    # 可選：應用遮罩 Apply mask (for decoder)
    if mask is not None:
        scores = scores + mask  # mask contains -inf for blocked positions
    
    # Softmax
    weights = softmax(scores, axis=-1)  # (seq_len, seq_len)
    
    # 加權求和 Weighted sum
    output = weights @ V  # (seq_len, d_v)
    
    return output, weights

# 模擬一個小的「詞嵌入」範例
np.random.seed(42)
words = ['The', 'cat', 'sat', 'on', 'the', 'mat']
seq_len_sa = len(words)
d_model_sa = 16
d_k = 8

# 模擬詞嵌入 (Simulated word embeddings)
X_embed = np.random.randn(seq_len_sa, d_model_sa)

# 投影矩陣 Projection matrices
W_Q = np.random.randn(d_model_sa, d_k) * 0.3
W_K = np.random.randn(d_model_sa, d_k) * 0.3
W_V = np.random.randn(d_model_sa, d_k) * 0.3

# 計算 Q, K, V
Q_sa = X_embed @ W_Q  # (6, 8)
K_sa = X_embed @ W_K  # (6, 8)
V_sa = X_embed @ W_V  # (6, 8)

# 計算 Self-Attention
output_sa, weights_sa = scaled_dot_product_attention(Q_sa, K_sa, V_sa)

print(f'Input shape: {X_embed.shape}')
print(f'Q, K, V shape: {Q_sa.shape}')
print(f'Attention weights shape: {weights_sa.shape}')
print(f'Output shape: {output_sa.shape}')

In [ ]:
# 視覺化 Self-Attention 矩陣
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：完整注意力矩陣
im1 = axes[0].imshow(weights_sa, cmap='Blues', aspect='auto')
axes[0].set_xticks(range(seq_len_sa))
axes[0].set_xticklabels(words, fontsize=11)
axes[0].set_yticks(range(seq_len_sa))
axes[0].set_yticklabels(words, fontsize=11)
axes[0].set_xlabel('Key (attends to)', fontsize=11)
axes[0].set_ylabel('Query (from)', fontsize=11)
axes[0].set_title('Self-Attention Matrix\n(Full / Encoder)', fontsize=12)
for i in range(seq_len_sa):
    for j in range(seq_len_sa):
        axes[0].text(j, i, f'{weights_sa[i, j]:.2f}',
                     ha='center', va='center', fontsize=8,
                     color='white' if weights_sa[i, j] > 0.25 else 'black')
plt.colorbar(im1, ax=axes[0], label='Weight')

# 右圖：Masked Self-Attention (causal mask for decoder)
causal_mask = np.triu(np.ones((seq_len_sa, seq_len_sa)) * (-1e9), k=1)
_, weights_masked = scaled_dot_product_attention(Q_sa, K_sa, V_sa, mask=causal_mask)

im2 = axes[1].imshow(weights_masked, cmap='Oranges', aspect='auto')
axes[1].set_xticks(range(seq_len_sa))
axes[1].set_xticklabels(words, fontsize=11)
axes[1].set_yticks(range(seq_len_sa))
axes[1].set_yticklabels(words, fontsize=11)
axes[1].set_xlabel('Key (attends to)', fontsize=11)
axes[1].set_ylabel('Query (from)', fontsize=11)
axes[1].set_title('Masked Self-Attention\n(Causal / Decoder)', fontsize=12)
for i in range(seq_len_sa):
    for j in range(seq_len_sa):
        axes[1].text(j, i, f'{weights_masked[i, j]:.2f}',
                     ha='center', va='center', fontsize=8,
                     color='white' if weights_masked[i, j] > 0.25 else 'black')
plt.colorbar(im2, ax=axes[1], label='Weight')

plt.suptitle('Self-Attention: Full vs Causal Masking\n自注意力：完整 vs 因果遮罩', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

print('左圖（Encoder）：每個位置可以關注所有位置')
print('右圖（Decoder）：每個位置只能關注自己和之前的位置')
print('  → 右上三角為 0，防止「作弊」看到未來的 token')

---
## Part 9: Transformer 架構概覽 Transformer Architecture Diagram

Transformer 的革命性主張：**完全捨棄遞迴結構，僅使用注意力機制**。

核心組件：
- **Multi-Head Attention**：多個注意力頭捕捉不同關係
- **Positional Encoding**：注入位置資訊
- **Feed-Forward Network**：逐位置的全連接層
- **Add & Norm**：殘差連接 + 層正規化

In [ ]:
# 繪製 Transformer 架構概覽圖
fig, ax = plt.subplots(figsize=(16, 14))
ax.set_xlim(0, 20)
ax.set_ylim(0, 18)
ax.axis('off')

def draw_block(ax, x, y, w, h, text, color, fontsize=9):
    """Draw a rounded rectangle block."""
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                          facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, fontsize=fontsize,
            ha='center', va='center', fontweight='bold')

# ====== Encoder Side (Left) ======
ax.text(5, 17.2, 'Encoder', fontsize=16, fontweight='bold', ha='center', color='#2C3E50')
ax.text(5, 16.7, '(x N layers)', fontsize=10, ha='center', color='gray')

# Encoder blocks (bottom to top)
draw_block(ax, 2.5, 1, 5, 1, 'Input Embedding\n+ Positional Encoding', '#E8DAEF', 9)
draw_block(ax, 2.5, 3, 5, 1.5, 'Multi-Head\nSelf-Attention', '#AED6F1', 10)
draw_block(ax, 2.5, 5, 5, 0.8, 'Add & LayerNorm', '#D5F5E3', 9)
draw_block(ax, 2.5, 6.3, 5, 1.2, 'Feed-Forward\nNetwork (FFN)', '#FADBD8', 10)
draw_block(ax, 2.5, 8, 5, 0.8, 'Add & LayerNorm', '#D5F5E3', 9)

# Encoder arrows
ax.annotate('', xy=(5, 3), xytext=(5, 2), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(5, 5), xytext=(5, 4.5), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(5, 6.3), xytext=(5, 5.8), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(5, 8), xytext=(5, 7.5), arrowprops=dict(arrowstyle='->', lw=1.5))

# Residual connections (encoder)
ax.annotate('', xy=(2.2, 5.4), xytext=(2.2, 3),
            arrowprops=dict(arrowstyle='->', lw=1, color='gray', ls='--'))
ax.annotate('', xy=(2.2, 8.4), xytext=(2.2, 6.3),
            arrowprops=dict(arrowstyle='->', lw=1, color='gray', ls='--'))

# Repeat bracket
ax.add_patch(FancyBboxPatch((1.5, 2.7), 7.2, 6.5, boxstyle='round,pad=0.1',
                             facecolor='none', edgecolor='gray', linewidth=1, linestyle='--'))

# ====== Decoder Side (Right) ======
ax.text(15, 17.2, 'Decoder', fontsize=16, fontweight='bold', ha='center', color='#2C3E50')
ax.text(15, 16.7, '(x N layers)', fontsize=10, ha='center', color='gray')

# Decoder blocks
draw_block(ax, 12.5, 1, 5, 1, 'Output Embedding\n+ Positional Encoding', '#E8DAEF', 9)
draw_block(ax, 12.5, 3, 5, 1.5, 'Masked Multi-Head\nSelf-Attention', '#F9E79F', 10)
draw_block(ax, 12.5, 5, 5, 0.8, 'Add & LayerNorm', '#D5F5E3', 9)
draw_block(ax, 12.5, 6.3, 5, 1.5, 'Multi-Head\nCross-Attention', '#AED6F1', 10)
draw_block(ax, 12.5, 8.3, 5, 0.8, 'Add & LayerNorm', '#D5F5E3', 9)
draw_block(ax, 12.5, 9.6, 5, 1.2, 'Feed-Forward\nNetwork (FFN)', '#FADBD8', 10)
draw_block(ax, 12.5, 11.3, 5, 0.8, 'Add & LayerNorm', '#D5F5E3', 9)

# Decoder arrows
ax.annotate('', xy=(15, 3), xytext=(15, 2), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(15, 5), xytext=(15, 4.5), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(15, 6.3), xytext=(15, 5.8), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(15, 8.3), xytext=(15, 7.8), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(15, 9.6), xytext=(15, 9.1), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(15, 11.3), xytext=(15, 10.8), arrowprops=dict(arrowstyle='->', lw=1.5))

# Cross-attention arrow from encoder to decoder
ax.annotate('', xy=(12.5, 7.05), xytext=(7.5, 8.4),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#E74C3C'))
ax.text(10, 8.3, 'K, V from\nEncoder', fontsize=9, ha='center', color='#E74C3C',
        fontstyle='italic')

# Repeat bracket
ax.add_patch(FancyBboxPatch((11.5, 2.7), 7.2, 9.8, boxstyle='round,pad=0.1',
                             facecolor='none', edgecolor='gray', linewidth=1, linestyle='--'))

# Output
draw_block(ax, 12.5, 12.8, 5, 0.8, 'Linear Layer', '#D7BDE2', 10)
draw_block(ax, 12.5, 14.2, 5, 0.8, 'Softmax', '#D2B4DE', 10)
ax.annotate('', xy=(15, 12.8), xytext=(15, 12.1), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(15, 14.2), xytext=(15, 13.6), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.text(15, 15.3, 'Output\nProbabilities', fontsize=11, ha='center', fontweight='bold')
ax.annotate('', xy=(15, 15.1), xytext=(15, 15), arrowprops=dict(arrowstyle='->', lw=1.5))

# Input labels
ax.text(5, 0.3, 'Source Input\n(源語言)', fontsize=10, ha='center')
ax.text(15, 0.3, 'Target Input (shifted)\n(目標語言)', fontsize=10, ha='center')

plt.title('Transformer Architecture Overview\nTransformer 架構概覽',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# 位置編碼視覺化 Positional Encoding Visualization
def positional_encoding(max_len, d_model):
    """
    生成正弦位置編碼 Generate sinusoidal positional encoding.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len).reshape(-1, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    
    pe[:, 0::2] = np.sin(position * div_term)  # 偶數維度
    pe[:, 1::2] = np.cos(position * div_term)  # 奇數維度
    return pe

# 生成位置編碼
max_len_pe = 50
d_model_pe = 64
pe = positional_encoding(max_len_pe, d_model_pe)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 上圖：位置編碼熱力圖
im = axes[0].imshow(pe.T, aspect='auto', cmap='RdBu_r',
                     interpolation='nearest')
axes[0].set_xlabel('Position (序列位置)', fontsize=11)
axes[0].set_ylabel('Dimension (編碼維度)', fontsize=11)
axes[0].set_title('Sinusoidal Positional Encoding\n正弦位置編碼', fontsize=13)
plt.colorbar(im, ax=axes[0], label='Value')

# 下圖：選取幾個維度的波形
dims_to_show = [0, 1, 4, 5, 10, 11, 20, 21]
for dim in dims_to_show:
    label = f'dim {dim} ({"sin" if dim % 2 == 0 else "cos"})'
    axes[1].plot(pe[:, dim], label=label, linewidth=1.2, alpha=0.8)
axes[1].set_xlabel('Position', fontsize=11)
axes[1].set_ylabel('Encoding Value', fontsize=11)
axes[1].set_title('Selected Dimensions of Positional Encoding\n位置編碼的部分維度波形', fontsize=13)
axes[1].legend(fontsize=8, ncol=4, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('位置編碼設計巧思 Design insights:')
print('  1. 低維度用高頻 → 區分相鄰位置')
print('  2. 高維度用低頻 → 編碼遠距離位置關係')
print('  3. PE(pos+k) 可表示為 PE(pos) 的線性函數 → 學到相對位置')
print('  4. 可外推到訓練時未見過的序列長度')

In [ ]:
# 比較表格：RNN → LSTM/GRU → Transformer 的技術演進
print('=' * 75)
print('從 RNN 到 Transformer 的技術演進 Evolution from RNN to Transformer')
print('=' * 75)

evolution = [
    ('Year', 'Architecture', 'Key Innovation'),
    ('----', '------------', '--------------'),
    ('1990', 'Simple RNN (Elman)', 'Hidden state + parameter sharing'),
    ('1997', 'LSTM', 'Cell state + 3 gates → solve vanishing gradient'),
    ('2014', 'GRU', 'Simplified LSTM with 2 gates'),
    ('2014', 'Seq2Seq + Attention', 'Break fixed-vector bottleneck'),
    ('2017', 'Transformer', 'Pure attention, no recurrence, parallelizable'),
    ('2018+', 'BERT / GPT', 'Large-scale pre-training on Transformer'),
]

for row in evolution:
    print(f'{row[0]:<8} {row[1]:<22} {row[2]}')

print('\n' + '=' * 75)
print(f'{"Property":<30} {"Vanilla RNN":<15} {"LSTM/GRU":<15} {"Transformer":<15}')
print('=' * 75)
props = [
    ('Long-range dependency', 'Poor', 'Good', 'Excellent'),
    ('Parallelization', 'None', 'None', 'Full'),
    ('Complexity (seq len n)', 'O(n)', 'O(n)', 'O(n^2)'),
    ('Max path length', 'O(n)', 'O(n)', 'O(1)'),
    ('Memory mechanism', 'Hidden state', 'Gated cells', 'Global attention'),
]
for prop, rnn, lstm, trans in props:
    print(f'{prop:<30} {rnn:<15} {lstm:<15} {trans:<15}')
print('=' * 75)

---
## Exercises 練習題

完成以下練習來鞏固本週所學。

### Exercise 1: 修改 RNN 以處理變長序列

修改 `VanillaRNNCell` 使其可以處理不同長度的輸入序列，
並在最後一個有效步驟返回隱藏狀態作為序列表示。

提示：接受一個 `lengths` 參數，在迴圈中根據序列長度停止更新。

In [ ]:
# Exercise 1: 請在此撰寫你的程式碼
# TODO: Modify VanillaRNNCell to handle variable-length sequences

# class VanillaRNNVariable(VanillaRNNCell):
#     def forward_variable(self, x_sequence, length):
#         """Process only the first 'length' steps of x_sequence."""
#         # Your code here
#         pass

# Test with sequences of different lengths:
# seq1 = np.random.randn(20, 1)  # length 20
# seq2 = np.random.randn(20, 1)  # length 15 (pad the rest)
# rnn_var = VanillaRNNVariable(1, 8, 1)
# h1, _ = rnn_var.forward_variable(seq1, length=20)
# h2, _ = rnn_var.forward_variable(seq2, length=15)

### Exercise 2: 視覺化 LSTM Forget Gate 對不同模式的反應

使用 Part 4 的 LSTM 實作，分別輸入：
1. 常數序列 (constant)
2. 脈衝序列 (pulse/spike)
3. 遞增序列 (ramp)

觀察並比較 Forget Gate 的激活模式有何不同。

In [ ]:
# Exercise 2: 請在此撰寫你的程式碼
# TODO: Compare LSTM forget gate activations across different input patterns

# Hint: Create three input sequences
# constant_input = np.ones(60) * 0.5
# pulse_input = np.zeros(60); pulse_input[10] = 3.0; pulse_input[30] = -3.0
# ramp_input = np.linspace(0, 2, 60)

# Run each through the LSTM and plot the forget gate values

### Exercise 3: 實作簡單的位置編碼

實作一個可學習的 (learnable) 位置編碼方案：
1. 生成一個隨機初始化的位置編碼矩陣 (max_len x d_model)
2. 將它與 Part 9 的正弦位置編碼進行視覺化比較
3. 計算兩個位置之間的餘弦相似度，驗證正弦編碼是否能編碼相對距離

In [ ]:
# Exercise 3: 請在此撰寫你的程式碼
# TODO: Implement and compare positional encoding schemes

# 1. Create learnable PE (random initialization)
# learnable_pe = np.random.randn(max_len_pe, d_model_pe) * 0.02

# 2. Compare visually with sinusoidal PE from Part 9

# 3. Compute cosine similarity between position pairs
#    for the sinusoidal PE and check if nearby positions
#    have higher similarity.
# def cosine_similarity(a, b):
#     return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
# sim_matrix = np.zeros((max_len_pe, max_len_pe))
# for i in range(max_len_pe):
#     for j in range(max_len_pe):
#         sim_matrix[i, j] = cosine_similarity(pe[i], pe[j])

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **序列資料生成**：正弦波、趨勢、方波、隨機遊走
2. **Vanilla RNN 實作**：觀察隱藏狀態如何編碼序列資訊
3. **梯度消失問題**：連乘矩陣導致梯度指數衰減/爆炸
4. **LSTM 門控結構**：三門控 + 細胞狀態的「高速公路」設計
5. **GRU vs LSTM**：簡化架構，參數更少，效能相當
6. **序列分類**：手工特徵 + sklearn 分類器
7. **點積注意力**：讓模型學會「看哪裡」
8. **Self-Attention**：Q, K, V 框架，全局依賴捕捉
9. **Transformer 架構**：純注意力，可平行化，O(1) 路徑長度

### 關鍵收穫 Key Takeaways
- RNN 引入了「記憶」但受制於梯度消失
- LSTM/GRU 透過**門控 + 加法更新**解決長程依賴
- Attention 讓模型**動態關注**相關資訊
- Transformer 完全拋棄遞迴，靠 Self-Attention 實現**平行化 + 全局依賴**

### 下週預告 Next Week Preview
**第 14 週：深度學習訓練技巧 (DL Training Tricks)**
- 學習率排程 (LR Scheduling)
- 早停 (Early Stopping)
- 資料增強 (Data Augmentation)
- 梯度裁剪 (Gradient Clipping)